# Natural language processing and large language models

*Data Science and AI, Module 8. Slides: `Module_8.pdf`.*

Keep the slides open on the right. Every heading below carries the slide
numbers it goes with, so the two stay in step.

Natural language processing (NLP) is about getting a computer to do useful
work with text people wrote for other people. By the end of this notebook
you will have taken text apart with patterns, pulled it off a web page,
cleaned it, scored its mood, trained a model to sort it, and looked inside
the dice roll that makes a large language model (LLM) write.

**How each section works.** A question comes first. Make a guess out loud or
in the chat before the cell runs. Then run it and compare.

**Four kinds of text.**

- Small **made-up** examples, always labelled as made up.
- The **Titanic passenger list** from the ML challenge (`titanic_train.csv`),
  because the names hide something useful.
- **2,000 film reviews**, 1,000 positive and 1,000 negative, from Bo Pang and
  Lillian Lee, "A Sentimental Education", Proceedings of the ACL, 2004
  (polarity dataset v2.0, distributed with NLTK with the authors'
  permission).
- One live web page, `books.toscrape.com`, a site built for scraping
  practice. Its prices are random and mean nothing.

The notebook is in six parts, one per section of the deck. Each part loads
what it needs, so you can start at any part.

## Before the first session: one-off downloads

The `dsai` environment already has NLTK and spaCy. Both need some data
files the first time. Run the cell below once; afterwards it takes a second.

spaCy's English model is installed from a terminal, not from Python. If the
cell says it is missing, open the Anaconda Prompt (Windows) or Terminal
(Mac), run `conda activate dsai`, then the command the cell prints.

In [1]:
import nltk
import spacy

# Each download is skipped quickly if it is already on this computer.
nltk.download("vader_lexicon", quiet=True)   # word list for sentiment (Part 4)
nltk.download("stopwords", quiet=True)       # common words (Part 3)
nltk.download("wordnet", quiet=True)         # dictionary for lemmas (Part 3)
nltk.download("movie_reviews", quiet=True)   # the 2,000 film reviews

print("nltk", nltk.__version__)
print("spacy", spacy.__version__)

if spacy.util.is_package("en_core_web_sm"):
    print("spaCy English model: installed")
else:
    print("spaCy English model missing. Run this once in a terminal:")
    print("    python -m spacy download en_core_web_sm")    

nltk 3.10.2
spacy 3.8.14
spaCy English model: installed


# Part 6: Large language models (LLMs)

Three ideas, no API key needed (Lab 8.5 makes real calls):

1. **How an LLM writes**: one token at a time, picked from probabilities
2. **Prompts**: a prompt is just a string you build
3. **RAG**: find the right documents first, then paste them into the prompt

All model replies in this part are **simulated**.

In [2]:
import math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## How an LLM writes

An LLM is trained to do one thing: given the text so far, give a **probability**
to every possible next **token** (a word or a piece of a word).
It picks one, adds it to the text, and repeats.

So the model writes what is **likely**, which is not always what is **true**.
That is where **hallucinations** come from.


The examples show the model the task **and** the answer format.

**Kinds of prompt:**
- **zero-shot**: instruction + review, no examples
- **chain of thought**: add "work through it step by step"
- **structured**: state the exact output format, e.g. "reply in JSON with keys `label` and `reason`"
- **iterative**: read the reply, improve the instruction, try again



## RAG: retrieval-augmented generation

An LLM only knows its training data. RAG adds a step:

1. **Search** your own documents for the ones that match the question
2. **Paste** them into the prompt
3. The LLM answers from the text it was given

The search step can use TF-IDF from Part 5. Below are **made-up** help pages for a bike-share scheme.

In [3]:
# Made up: help pages for a bike-share scheme
help_pages = [
    "Pricing: a single ride is 3 dollars for the first 30 minutes, then 1 dollar per extra 15 minutes.",
    "Membership: an annual membership is 90 dollars and includes unlimited 45 minute rides.",
    "Lost bike: report a lost or stolen bike in the app within 24 hours to avoid a 500 dollar fee.",
    "Docking: always check the green light on the dock. A bike not docked properly keeps the ride running.",
    "Safety: helmets are required by law. Lights must be on between sunset and sunrise.",
    "Students: students with a valid ID get 30 percent off the annual membership.",
]

# Turn every page into TF-IDF numbers (same tool as Part 5)
page_vectoriser = TfidfVectorizer(stop_words="english")
page_numbers = page_vectoriser.fit_transform(help_pages)



### Step 1: Search

**Cosine similarity** measures how much the question and a page share:
0 = no words in common, 1 = identical.

**Before you run it:** which two pages should come back for this question?

In [4]:
# The user's question
question = "I am a student. How much is a membership for the year?"

# Step 1: turn the question into numbers with the SAME vectoriser
question_numbers = page_vectoriser.transform([question])

# Step 2: how similar is the question to each page? (0 = nothing shared, 1 = identical)
similarities = cosine_similarity(question_numbers, page_numbers)[0]

# Step 3: put pages and scores in a table, best match first
results = pd.DataFrame({"similarity": similarities.round(3), "page": help_pages})
results = results.sort_values("similarity", ascending=False)

print(results.to_string(index=False))

 similarity                                                                                                  page
      0.518                Membership: an annual membership is 90 dollars and includes unlimited 45 minute rides.
      0.273                          Students: students with a valid ID get 30 percent off the annual membership.
      0.000     Pricing: a single ride is 3 dollars for the first 30 minutes, then 1 dollar per extra 15 minutes.
      0.000         Lost bike: report a lost or stolen bike in the app within 24 hours to avoid a 500 dollar fee.
      0.000 Docking: always check the green light on the dock. A bike not docked properly keeps the ride running.
      0.000                    Safety: helmets are required by law. Lights must be on between sunset and sunrise.


### When the search fails

Same question idea, different words. **Before you run it:** which page should come first?

In [5]:
# Same steps, a question worded differently
question = "What does it cost to hire a bicycle once?"

question_numbers = page_vectoriser.transform([question])
similarities = cosine_similarity(question_numbers, page_numbers)[0]

results = pd.DataFrame({"similarity": similarities.round(3), "page": help_pages})
results = results.sort_values("similarity", ascending=False)

print(results.to_string(index=False))

 similarity                                                                                                  page
        0.0     Pricing: a single ride is 3 dollars for the first 30 minutes, then 1 dollar per extra 15 minutes.
        0.0                Membership: an annual membership is 90 dollars and includes unlimited 45 minute rides.
        0.0         Lost bike: report a lost or stolen bike in the app within 24 hours to avoid a 500 dollar fee.
        0.0 Docking: always check the green light on the dock. A bike not docked properly keeps the ride running.
        0.0                    Safety: helmets are required by law. Lights must be on between sunset and sunrise.
        0.0                          Students: students with a valid ID get 30 percent off the annual membership.


"cost", "hire" and "bicycle" are not on the pricing page ("price", "ride", "bike" are),
so TF-IDF finds nothing. The LLM would get no useful information.


## Your turn

Copy Code 10 and change the question:

1. Write a question that finds the right page.
2. Write a question that **should** find a page but doesn't. Why did it fail?

## What to take away

- An LLM picks each token from probabilities. Temperature and top_p change **how** it picks, not **what it knows**.
- A prompt is a string. Test prompts like models: same examples, count the results.
- RAG = search + prompt. If the search finds the wrong pages, the answer is built on the wrong text.

In [6]:
question = "Someone stole my bike, how do I report it?"
# Step 1: turn the question into numbers with the SAME vectoriser
question_numbers = page_vectoriser.transform([question])
# Step 2: compare the question with every page
all_scores = cosine_similarity(question_numbers, page_numbers)[0]
